In [1]:
!pip install pandas
!pip install bs4


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
from pathlib import Path

path = r"..\archive\Sarcasm_Headlines_Dataset_v2.json"
df = pd.read_json(path, lines=True)

print(df.head())
print(df.shape)
print(df.columns)

   is_sarcastic                                           headline  \
0             1  thirtysomething scientists unveil doomsday clo...   
1             0  dem rep. totally nails why congress is falling...   
2             0  eat your veggies: 9 deliciously different recipes   
3             1  inclement weather prevents liar from getting t...   
4             1  mother comes pretty close to using word 'strea...   

                                        article_link  
0  https://www.theonion.com/thirtysomething-scien...  
1  https://www.huffingtonpost.com/entry/donna-edw...  
2  https://www.huffingtonpost.com/entry/eat-your-...  
3  https://local.theonion.com/inclement-weather-p...  
4  https://www.theonion.com/mother-comes-pretty-c...  
(28619, 3)
Index(['is_sarcastic', 'headline', 'article_link'], dtype='str')


In [4]:
import json
import threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urlsplit, urlunsplit

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from tqdm import tqdm
from urllib3.util.retry import Retry

CDX_URL = "https://web.archive.org/cdx/search/cdx"

SNAPSHOT_COLS = [
    "wayback_available",
    "wayback_url",
    "wayback_timestamp",
    "wayback_status",
    "wayback_error",
]

CONTENT_COLS = [
    "archived_title",
    "archived_meta_description",
    "archived_article_text",
    "content_error",
]

ALL_COLS = SNAPSHOT_COLS + CONTENT_COLS
_thread_local = threading.local()


# -----------------------------
# Cache helpers
# -----------------------------
def _save_json(path: Path, obj: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=True)
    tmp.replace(path)

def _load_json(path: Path):
    if not path.exists():
        return {}
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return data if isinstance(data, dict) else {}
    except Exception:
        return {}


# -----------------------------
# HTTP helpers
# -----------------------------
def build_session(retries: int = 5, backoff_factor: float = 1.0):
    s = requests.Session()
    retry = Retry(
        total=retries,
        connect=retries,
        read=retries,
        backoff_factor=backoff_factor,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=50, pool_maxsize=50)
    s.mount("http://", adapter)
    s.mount("https://", adapter)
    s.headers.update({"User-Agent": "Mozilla/5.0"})
    return s

def get_session(retries: int = 5, backoff_factor: float = 1.0):
    if (
        not hasattr(_thread_local, "session")
        or getattr(_thread_local, "retries", None) != retries
        or getattr(_thread_local, "backoff_factor", None) != backoff_factor
    ):
        _thread_local.session = build_session(retries=retries, backoff_factor=backoff_factor)
        _thread_local.retries = retries
        _thread_local.backoff_factor = backoff_factor
    return _thread_local.session

def safe_get(url, params=None, timeout=(5, 25), retries: int = 5, backoff_factor: float = 1.0):
    r = get_session(retries=retries, backoff_factor=backoff_factor).get(url, params=params, timeout=timeout)
    r.raise_for_status()
    return r


# -----------------------------
# Wayback helpers
# -----------------------------
def clean_text(text: str) -> str:
    return " ".join(text.split()) if text else ""

def empty_snapshot():
    return {
        "wayback_available": False,
        "wayback_url": None,
        "wayback_timestamp": None,
        "wayback_status": None,
        "wayback_error": None,
    }

def empty_content():
    return {
        "archived_title": None,
        "archived_meta_description": None,
        "archived_article_text": None,
        "content_error": None,
    }

def url_variants(url: str):
    if not isinstance(url, str) or not url.strip():
        return []
    u = url.strip()
    p = urlsplit(u)
    netloc = p.netloc.lower()
    path = p.path or "/"

    variants = set()
    for keep_query in [True, False]:
        q = p.query if keep_query else ""
        for scheme in ["https", "http"]:
            variants.add(urlunsplit((scheme, netloc, path, q, "")))
            alt = (path[:-1] or "/") if path.endswith("/") else (path + "/")
            variants.add(urlunsplit((scheme, netloc, alt, q, "")))

    for v in list(variants):
        pv = urlsplit(v)
        n = pv.netloc
        n2 = n[4:] if n.startswith("www.") else "www." + n
        variants.add(urlunsplit((pv.scheme, n2, pv.path, pv.query, "")))

    return [u] + [x for x in variants if x != u]

def find_snapshot(url: str, retries: int = 5, backoff_factor: float = 1.0):
    out = empty_snapshot()
    if not isinstance(url, str) or not url.strip():
        out["wayback_error"] = "Invalid/empty URL"
        return out

    try:
        for candidate in url_variants(url):
            params_strict = {
                "url": candidate,
                "output": "json",
                "fl": "timestamp,original,statuscode,mimetype",
                "filter": ["statuscode:200", "mimetype:text/html"],
                "collapse": "digest",
                "from": "2000",
                "to": "2026",
            }
            r = safe_get(CDX_URL, params=params_strict, timeout=(5, 25), retries=retries, backoff_factor=backoff_factor)
            data = r.json()
            if len(data) > 1:
                ts, original, status, _ = data[-1]
                out.update({
                    "wayback_available": True,
                    "wayback_url": f"https://web.archive.org/web/{ts}/{original}",
                    "wayback_timestamp": ts,
                    "wayback_status": status,
                    "wayback_error": None,
                })
                return out

            params_relaxed = {
                "url": candidate,
                "output": "json",
                "fl": "timestamp,original,statuscode,mimetype",
                "filter": ["statuscode:200"],
                "collapse": "digest",
                "from": "2000",
                "to": "2026",
            }
            r2 = safe_get(CDX_URL, params=params_relaxed, timeout=(5, 25), retries=retries, backoff_factor=backoff_factor)
            data2 = r2.json()
            if len(data2) > 1:
                ts, original, status, _ = data2[-1]
                out.update({
                    "wayback_available": True,
                    "wayback_url": f"https://web.archive.org/web/{ts}/{original}",
                    "wayback_timestamp": ts,
                    "wayback_status": status,
                    "wayback_error": None,
                })
                return out

        return out
    except Exception as e:
        out["wayback_error"] = f"{type(e).__name__}: {e}"
        return out

def extract_content(wayback_url: str, retries: int = 5, backoff_factor: float = 1.0, max_chars: int = 4000):
    out = empty_content()
    if not isinstance(wayback_url, str) or not wayback_url.strip():
        out["content_error"] = "Invalid/empty wayback_url"
        return out

    try:
        r = safe_get(wayback_url, timeout=(5, 40), retries=retries, backoff_factor=backoff_factor)
        soup = BeautifulSoup(r.text, "lxml")

        for tag in soup(["script", "style", "noscript", "header", "footer", "nav", "aside"]):
            tag.decompose()

        out["archived_title"] = clean_text(soup.title.get_text()) if soup.title else None

        meta_desc = ""
        m = soup.find("meta", attrs={"name": "description"})
        if m and m.get("content"):
            meta_desc = clean_text(m["content"])
        out["archived_meta_description"] = meta_desc or None

        selectors = ["article", "[role='main']", ".article", ".entry-content", ".post-content", ".content", "main"]
        candidates = []
        for sel in selectors:
            for node in soup.select(sel):
                txt = clean_text(node.get_text(" ", strip=True))
                if len(txt) > 200:
                    candidates.append(txt)

        if candidates:
            text = max(candidates, key=len)
        else:
            paras = [clean_text(p.get_text(" ", strip=True)) for p in soup.find_all("p")]
            paras = [p for p in paras if len(p) > 40]
            text = " ".join(paras[:20])

        out["archived_article_text"] = text[:max_chars] if text else None
        return out
    except Exception as e:
        out["content_error"] = f"{type(e).__name__}: {e}"
        return out


# -----------------------------
# Resume-aware main pipeline
# -----------------------------
def enrich_with_wayback_fast_resume_errors(
    df: pd.DataFrame,
    url_col: str = "article_link",
    workers_snapshot: int = 12,
    workers_content: int = 8,
    retries: int = 5,
    backoff_factor: float = 1.0,
    cache_dir: str = "./wayback_cache",
    checkpoint_csv: str = "headline_with_wayback_context.csv",
    save_every_n_urls: int = 500,
    rerun_errors: bool = True,
):
    if url_col not in df.columns:
        raise ValueError(f"DataFrame must contain '{url_col}'")

    cache_dir = Path(cache_dir)
    snapshot_cache_path = cache_dir / "snapshot_cache.json"
    content_cache_path = cache_dir / "content_cache.json"

    snapshot_cache = _load_json(snapshot_cache_path)
    content_cache = _load_json(content_cache_path)

    unique_urls = [u for u in df[url_col].dropna().astype(str).unique().tolist() if u.strip()]

    # Snapshot todo:
    # - missing cache entry, OR
    # - has wayback_error and rerun_errors=True
    snapshot_todo = []
    for u in unique_urls:
        s = snapshot_cache.get(u)
        if s is None:
            snapshot_todo.append(u)
        elif rerun_errors and s.get("wayback_error"):
            snapshot_todo.append(u)

    print(f"[INFO] unique_urls={len(unique_urls)}")
    print(f"[INFO] snapshot_cache entries={len(snapshot_cache)} | content_cache entries={len(content_cache)}")
    print(f"[INFO] snapshot_todo={len(snapshot_todo)} (missing + error reruns)")

    # PASS 1: Snapshot
    run_hit = run_miss = run_err = 0
    for i in range(0, len(snapshot_todo), save_every_n_urls):
        chunk = snapshot_todo[i:i + save_every_n_urls]

        with ThreadPoolExecutor(max_workers=workers_snapshot) as ex:
            fut_to_url = {ex.submit(find_snapshot, u, retries, backoff_factor): u for u in chunk}
            pbar = tqdm(total=len(chunk), desc=f"snapshot {i}:{i+len(chunk)}")

            for fut in as_completed(fut_to_url):
                u = fut_to_url[fut]
                try:
                    res = fut.result()
                except Exception as e:
                    res = empty_snapshot()
                    res["wayback_error"] = f"{type(e).__name__}: {e}"

                snapshot_cache[u] = res

                if res.get("wayback_error"):
                    run_err += 1
                elif res.get("wayback_available"):
                    run_hit += 1
                else:
                    run_miss += 1

                pbar.update(1)
                pbar.set_postfix(hit=run_hit, miss=run_miss, err=run_err, refresh=False)

            pbar.close()

        _save_json(snapshot_cache_path, snapshot_cache)
        print(f"[SNAPSHOT SAVE] {snapshot_cache_path} | hit={run_hit} miss={run_miss} err={run_err}")

    # Content todo:
    # - only URLs that currently have snapshot hits
    # - missing content cache, OR content_error when rerun_errors=True
    hit_urls = [u for u, s in snapshot_cache.items() if s.get("wayback_available") and s.get("wayback_url")]
    content_todo = []
    for u in hit_urls:
        c = content_cache.get(u)
        if c is None:
            content_todo.append(u)
        elif rerun_errors and c.get("content_error"):
            content_todo.append(u)

    print(f"[INFO] hit_urls={len(hit_urls)}")
    print(f"[INFO] content_todo={len(content_todo)} (missing + error reruns)")

    # PASS 2: Content
    content_ok = content_err = 0
    for i in range(0, len(content_todo), save_every_n_urls):
        chunk = content_todo[i:i + save_every_n_urls]

        def _extract_for_url(u):
            wb_url = snapshot_cache[u]["wayback_url"]
            return extract_content(wb_url, retries=retries, backoff_factor=backoff_factor)

        with ThreadPoolExecutor(max_workers=workers_content) as ex:
            fut_to_url = {ex.submit(_extract_for_url, u): u for u in chunk}
            pbar = tqdm(total=len(chunk), desc=f"content {i}:{i+len(chunk)}")

            for fut in as_completed(fut_to_url):
                u = fut_to_url[fut]
                try:
                    c = fut.result()
                except Exception as e:
                    c = empty_content()
                    c["content_error"] = f"{type(e).__name__}: {e}"

                content_cache[u] = c
                if c.get("content_error"):
                    content_err += 1
                else:
                    content_ok += 1

                pbar.update(1)
                pbar.set_postfix(ok=content_ok, err=content_err, refresh=False)

            pbar.close()

        _save_json(content_cache_path, content_cache)
        print(f"[CONTENT SAVE] {content_cache_path} | ok={content_ok} err={content_err}")

    # Merge
    rows = []
    for u in df[url_col].fillna("").astype(str).tolist():
        s = snapshot_cache.get(u, empty_snapshot())
        c = content_cache.get(u, empty_content()) if s.get("wayback_available") else empty_content()
        row = {k: s.get(k) for k in SNAPSHOT_COLS}
        row.update({k: c.get(k) for k in CONTENT_COLS})
        rows.append(row)

    enriched = pd.concat([df.reset_index(drop=True), pd.DataFrame(rows, columns=ALL_COLS)], axis=1)
    enriched.to_csv(checkpoint_csv, index=False)

    hits = int(enriched["wayback_available"].fillna(False).sum())
    misses = int((~enriched["wayback_available"].fillna(False)).sum())
    snap_errors = int(enriched["wayback_error"].notna().sum())
    content_errors = int(enriched["content_error"].notna().sum())

    print(f"[DONE] hits={hits} misses={misses} snapshot_errors={snap_errors} content_errors={content_errors}")
    print(f"[DONE] output={Path(checkpoint_csv).resolve()}")

    return enriched


In [ ]:
enriched_df = enrich_with_wayback_fast_resume_errors(
    df,
    url_col="article_link",
    workers_snapshot=12,
    workers_content=8,
    retries=5,
    backoff_factor=1.0,
    cache_dir="./wayback_cache",
    checkpoint_csv="headline_with_wayback_context.csv",
    save_every_n_urls=500,
    rerun_errors=False,  # this is the key behavior you asked for
)


[INFO] unique_urls=28617
[INFO] snapshot_cache entries=0 | content_cache entries=0
[INFO] snapshot_todo=28617 (missing + error reruns)


snapshot 0:500:  41%|████      | 204/500 [14:59<22:39,  4.59s/it, err=145, hit=55, miss=3]  